# ANSYS Journal File Editor

Create and edit ANSYS journal files (.jou) easily in this notebook. Write your journal commands, then export them directly as .jou files for use in ANSYS.

In [12]:
import os
from pathlib import Path
from datetime import datetime
import tempfile

# Configuration
DEFAULT_EXPORT_DIR = os.path.expanduser("~/Desktop")  # Change this to your preferred directory
CURRENT_PROJECT = "ANSYS_Journal"

# Create a sample template
SAMPLE_JOURNAL = """! ANSYS Journal File
! Created: {timestamp}
! Description: Your journal file description here

! Enter your ANSYS commands below

! Example commands:
! /PREP7
! ET,1,SOLID185
! BLOCK,0,10,0,10,0,10
! ESIZE,1
! VMESH,ALL
! /SOLU
! ANTYPE,STATIC
! SOLVE
! /POST1
"""

print("✓ ANSYS Journal File Editor loaded")
print(f"✓ Default export directory: {DEFAULT_EXPORT_DIR}")


✓ ANSYS Journal File Editor loaded
✓ Default export directory: C:\Users\<YOUR_USERNAME>/Desktop


## Configuration & Control Panel

Change all settings below before editing your journal file

In [13]:
# ============================================================
# CONFIGURATION - MODIFY THIS SECTION BEFORE EDITING YOUR JOURNAL
# ============================================================

# OUTPUT SETTINGS
export_filename = "3.1.1.NG.0.20"  # Change this to your desired filename (no .jou needed)
export_directory = r"C:\Users\<YOUR_USERNAME>\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Journal Files"  # Where to save the .jou file

# QUICK ACTIONS - Choose what you want to do
load_existing_journal = False  # Set to True to load an existing .jou file
existing_journal_path = "C:/path/to/your/journal.jou"  # Path to load from (only used if above is True)

# After editing, you can:
#   1. Run the EXPORT cell to save your changes
#   2. Run VALIDATE to check for syntax issues
#   3. Run LIST FILES to see all your saved journals

print("=" * 70)
print("CONFIGURATION LOADED")
print("=" * 70)
print(f"📁 Export filename: {export_filename}.jou")
print(f"📁 Export directory: {export_directory}")
print(f"📋 Load existing file: {load_existing_journal}")
print("=" * 70)

CONFIGURATION LOADED
📁 Export filename: 3.1.1.NG.0.20.jou
📁 Export directory: C:\Users\<YOUR_USERNAME>\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Journal Files
📋 Load existing file: False


In [ ]:
# ============ EDIT YOUR JOURNAL CONTENT HERE ============
# Modify the text below with your ANSYS journal commands
# You can use multi-line strings or paste your journal code directly

journal_content = """; ============================================================
; ANSYS Fluent Journal File — AoA Sweep
; ============================================================

(define base-output-dir "C:/Users/<YOUR_USERNAME>/Documents/Thesis/NACA_2414_2D/Fluent/Directories/2414_006_005/3.1.1.NG/Case_and_Data")

; ---- Parameters ----
(define V_mag 24.38)
(define aoa_start 0)
(define aoa_end 1)
(define aoa_step 1)

; ---- Report file names ----
(define drag-report-file-name "drag-rfile")
(define lift-report-file-name "lift-rfile")

; ============================================================
; BOUNDARY CONDITIONS BY AoA
; ============================================================

; ---- AoA RANGE THRESHOLDS ----
; These define where each range STARTS.
; Example: threshold1 = 5 means Range 2 starts at AoA 5
;          So Range 1 = AoA 0-4, Range 2 = AoA 5-9, etc.
;
; To change ranges, just change the number:
;   threshold1 = 6  -->  Range 1 = AoA 0-5, Range 2 = AoA 6-9
;   threshold1 = 8  -->  Range 1 = AoA 0-7, Range 2 = AoA 8-9
;
(define threshold1 6)   ; Range 2 starts at this AoA
(define threshold2 10)  ; Range 3 starts at this AoA
(define threshold3 15)  ; Range 4 starts at this AoA

; ---- ZONE LISTS (edit zone names as needed) ----

; Range 1: AoA 0 to (threshold1 - 1)
(define inlet-list-1 '("inlet"))
(define outlet-list-1 '("back_pressure_outlet" "top"))
(define symmetry-list-1 '("side_sym" "bottom"))

; Range 2: threshold1 to (threshold2 - 1)
(define inlet-list-2 '("inlet" "bottom"))
(define outlet-list-2 '("back_pressure_outlet" "top"))
(define symmetry-list-2 '("side_sym"))

; Range 3: threshold2 to (threshold3 - 1)
(define inlet-list-3 '("inlet" "bottom"))
(define outlet-list-3 '("back_pressure_outlet" "top"))
(define symmetry-list-3 '("side_sym"))

; Range 4: threshold3 to aoa_end
(define inlet-list-4 '("inlet" "bottom"))
(define outlet-list-4 '("back_pressure_outlet" "top"))
(define symmetry-list-4 '("side_sym"))

; ============================================================
; HELPER FUNCTIONS (no need to edit below)
; ============================================================

(define (deg-to-rad deg) (* deg (/ 3.14159265359 180.0)))

(define (ensure-directory dir-path)
  (system (format #f "mkdir \"~a\"" dir-path)))

(define (get-inlet-zones aoa)
  (cond ((< aoa threshold1) inlet-list-1)
        ((< aoa threshold2) inlet-list-2)
        ((< aoa threshold3) inlet-list-3)
        (else inlet-list-4)))

(define (get-outlet-zones aoa)
  (cond ((< aoa threshold1) outlet-list-1)
        ((< aoa threshold2) outlet-list-2)
        ((< aoa threshold3) outlet-list-3)
        (else outlet-list-4)))

(define (get-symmetry-zones aoa)
  (cond ((< aoa threshold1) symmetry-list-1)
        ((< aoa threshold2) symmetry-list-2)
        ((< aoa threshold3) symmetry-list-3)
        (else symmetry-list-4)))

; ---- Create base output directory ----
(ensure-directory base-output-dir)

; ============================================================
; MAIN LOOP
; ============================================================

(do ((aoa aoa_start (+ aoa aoa_step)))
    ((> aoa aoa_end))
  
  (define current-aoa-dir (format #f "~a/AoA_~a" base-output-dir aoa))
  (ensure-directory current-aoa-dir)
  
  (define aoa_rad (deg-to-rad aoa))
  (define V_x (* V_mag (cos aoa_rad)))
  (define V_y (* V_mag (sin aoa_rad)))
  (define V_z 0.0)
  
  (display (format #f "~%===== AoA = ~a° =====~%" aoa))
  
  ; ---- Update report file paths ----
  (define new-drag-path (format #f "~a/drag_force_AoA_~a.txt" current-aoa-dir aoa))
  (define new-lift-path (format #f "~a/lift_force_AoA_~a.txt" current-aoa-dir aoa))
  
  (ti-menu-load-string
    (format #f "/solve/report-files/edit ~a file-name \"~a\" q~%"
            drag-report-file-name new-drag-path))
  (ti-menu-load-string
    (format #f "/solve/report-files/edit ~a file-name \"~a\" q~%"
            lift-report-file-name new-lift-path))
  
  ; ---- Apply velocity inlet BCs ----
  (for-each
    (lambda (zone-name)
      (ti-menu-load-string
        (format #f "/define/boundary-conditions/velocity-inlet ~a no yes yes no 0 yes no ~a no ~a no ~a no no yes 0.05 10~%"
          zone-name V_x V_y V_z)))
    (get-inlet-zones aoa))
  
  ; ---- Apply pressure outlet BCs ----
  (for-each
    (lambda (zone-name)
      (ti-menu-load-string
        (format #f "/define/boundary-conditions/pressure-outlet ~a yes no 0 no yes no no yes 0.05 10 yes no no~%"
          zone-name)))
    (get-outlet-zones aoa))
  
  ; ---- Apply symmetry BCs ----
  (for-each
    (lambda (zone-name)
      (ti-menu-load-string
        (format #f "/define/boundary-conditions/symmetry ~a~%" zone-name)))
    (get-symmetry-zones(aoa)))
  
  ; ---- Run solver ----
  (ti-menu-load-string "/solve/iterate 5")
  
  ; ---- Save case and data ----
  (define case-data-name (format #f "3.1.1.NG.~a.cas.h5" aoa))
  (ti-menu-load-string
    (format #f "/file/write-case-data ~a/~a yes~%" current-aoa-dir case-data-name))
  
  (display (format #f "AoA = ~a° complete!~%~%" aoa)))

(display "~%=== AoA sweep completed! ===~%")

"""

# Display current content
print("Current journal content:")
print("=" * 60)
print(journal_content)
print("=" * 60)
print("\n" + "=" * 60)
print("HOW TO CUSTOMIZE")
print("=" * 60)
print("""
1. CHANGE AoA RANGE THRESHOLDS:
   (define threshold1 5)   ; Range 2 starts at AoA 5
   (define threshold2 10)  ; Range 3 starts at AoA 10
   (define threshold3 15)  ; Range 4 starts at AoA 15

2. CHANGE ZONE LISTS:
   (define inlet-list-1 '("inlet" "bottom"))
   (define outlet-list-1 '("outlet" "top"))
   (define symmetry-list-1 '("side_sym" "bottom"))
""")

Current journal content:
; ============================================================
; ANSYS Fluent Journal File — AoA Sweep
; ============================================================

(define base-output-dir "C:/Users/<YOUR_USERNAME>/Documents/Thesis/NACA_2414_2D/Fluent/Directories/2414_006_005/3.1.1.NG/Case_and_Data")

; ---- Parameters ----
(define V_mag 24.38)
(define aoa_start 0)
(define aoa_end 1)
(define aoa_step 1)

; ---- Report file names ----
(define drag-report-file-name "drag-rfile")
(define lift-report-file-name "lift-rfile")

; ============================================================
; BOUNDARY CONDITIONS BY AoA
; ============================================================

; ---- AoA RANGE THRESHOLDS ----
; These define where each range STARTS.
; Example: threshold1 = 5 means Range 2 starts at AoA 5
;          So Range 1 = AoA 0-4, Range 2 = AoA 5-9, etc.
;
; To change ranges, just change the number:
;   threshold1 = 6  -->  Range 1 = AoA 0-5, Range 2 = AoA 

## Export Journal File

Use the functions below to save your journal file as .jou

In [19]:
def export_journal(filename, content, directory=None):
    """
    Export journal content to a .jou file
    
    Parameters:
    -----------
    filename : str
        Name of the file (with or without .jou extension)
    content : str
        The journal content to export
    directory : str, optional
        Directory to save the file. If None, uses DEFAULT_EXPORT_DIR
    
    Returns:
    --------
    str : Full path to the exported file
    """
    if directory is None:
        directory = DEFAULT_EXPORT_DIR
    
    # Ensure directory exists
    os.makedirs(directory, exist_ok=True)
    
    # Add .jou extension if not present
    if not filename.lower().endswith('.jou'):
        filename = filename + '.jou'
    
    # Create full file path
    filepath = os.path.join(directory, filename)
    
    # Write content to file
    with open(filepath, 'w') as f:
        f.write(content)
    
    return filepath


def export_current_journal(filename, directory=None):
    """
    Export the current journal_content to a .jou file
    
    Parameters:
    -----------
    filename : str
        Name of the file (with or without .jou extension)
    directory : str, optional
        Directory to save the file
    """
    filepath = export_journal(filename, journal_content, directory)
    
    # Get file size
    file_size = os.path.getsize(filepath)
    
    print(f"✓ Journal file exported successfully!")
    print(f"  Filename: {os.path.basename(filepath)}")
    print(f"  Location: {filepath}")
    print(f"  Size: {file_size} bytes")
    print(f"\n✓ Ready to use in ANSYS!")
    
    return filepath


# Example usage - uncomment and modify the line below to export:
# export_current_journal("my_simulation")

print("✓ Export functions ready")
print("✓ To export your journal, run the next cell")


✓ Export functions ready
✓ To export your journal, run the next cell


In [34]:
# ============================================================
# QUICK ACTIONS - RUN THESE AFTER EDITING YOUR JOURNAL
# ============================================================

# EXPORT YOUR JOURNAL
print("\n" + "="*70)
print("EXPORTING JOURNAL FILE...")
print("="*70)
filepath = export_current_journal(export_filename, export_directory)

# Display quick reference
print("\n✓ YOUR JOURNAL IS READY!")
print(f"  File: {filepath}")
print(f"  You can now use this in ANSYS!")



EXPORTING JOURNAL FILE...
✓ Journal file exported successfully!
  Filename: 3.1.1.NG.0.20.jou
  Location: C:\Users\<YOUR_USERNAME>\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Journal Files\3.1.1.NG.0.20.jou
  Size: 5245 bytes

✓ Ready to use in ANSYS!

✓ YOUR JOURNAL IS READY!
  File: C:\Users\<YOUR_USERNAME>\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Journal Files\3.1.1.NG.0.20.jou
  You can now use this in ANSYS!


## Quick Reference - Load Files & Utilities (Optional)

In [ ]:
def load_journal(filepath):
    """
    Load an existing journal file
    
    Parameters:
    -----------
    filepath : str
        Path to the journal file
    
    Returns:
    --------
    str : Content of the journal file
    """
    if not os.path.exists(filepath):
        print(f"✗ File not found: {filepath}")
        return None
    
    with open(filepath, 'r') as f:
        content = f.read()
    
    file_size = os.path.getsize(filepath)
    lines = len(content.split('\n'))
    
    print(f"✓ Journal file loaded successfully!")
    print(f"  Filename: {os.path.basename(filepath)}")
    print(f"  Size: {file_size} bytes")
    print(f"  Lines: {lines}")
    
    return content


def list_journal_files(directory=None):
    """
    List all .jou files in a directory
    
    Parameters:
    -----------
    directory : str, optional
        Directory to search. If None, uses DEFAULT_EXPORT_DIR
    """
    if directory is None:
        directory = DEFAULT_EXPORT_DIR
    
    if not os.path.exists(directory):
        print(f"✗ Directory not found: {directory}")
        return []
    
    jou_files = [f for f in os.listdir(directory) if f.lower().endswith('.jou')]
    
    if not jou_files:
        print(f"No .jou files found in {directory}")
        return []
    
    print(f"Found {len(jou_files)} journal file(s) in {directory}:")
    print("-" * 60)
    
    for i, filename in enumerate(jou_files, 1):
        filepath = os.path.join(directory, filename)
        size = os.path.getsize(filepath)
        mod_time = datetime.fromtimestamp(os.path.getmtime(filepath)).strftime("%Y-%m-%d %H:%M:%S")
        print(f"{i:2}. {filename:<40} {size:>10} bytes  {mod_time}")
    
    print("-" * 60)
    return jou_files


# ============================================================
# USE THESE HELPERS BELOW
# ============================================================

# TO LOAD AN EXISTING JOURNAL:
# Step 1: Find your file
# list_journal_files("C:/path/to/folder")
#
# Step 2: Load it
# my_journal = load_journal("C:/path/to/your/journal.jou")
#
# Step 3: Use it (copy the content and paste into cell 3)
# print(my_journal)

print("✓ Utility functions ready")


## Tips & Advanced Features

In [ ]:
def validate_journal(content):
    """
    Basic validation of journal syntax
    """
    lines = content.strip().split('\n')
    
    print(f"✓ Journal validation results:")
    print(f"  Total lines: {len(lines)}")
    print(f"  Non-comment lines: {sum(1 for l in lines if l.strip() and not l.strip().startswith('!'))}")
    print(f"  Comment lines: {sum(1 for l in lines if l.strip().startswith('!'))}")
    print(f"  Empty lines: {sum(1 for l in lines if not l.strip())}")
    
    return True


# ============================================================
# VALIDATE YOUR JOURNAL BEFORE EXPORTING
# ============================================================

# Run this to check your journal:
# validate_journal(journal_content)

print("✓ Advanced features ready")
